# Vision System Explained — How Robot Perception Works

**Purpose:** Understand how the robot's perception system detects balls, obstacles, and baskets.

This notebook explains:
- HSV color space and color segmentation
- Ball detection (blue, red, silver)
- Obstacle detection (yellow tape, edges)
- Basket detection
- Distance estimation techniques

## 1. Setup

In [ ]:
import sys
import os
import cv2
import numpy as np
import yaml
from IPython.display import display, Image
import ipywidgets as widgets

# Auto-detect project root by searching for config.yaml marker
project_root = os.getcwd()
while not os.path.exists(os.path.join(project_root, 'config.yaml')) and project_root != '/':
    project_root = os.path.dirname(project_root)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.hardware.camera import CameraController
from src.perception.ball_detector import BallDetector
from src.perception.obstacle_detector import ObstacleDetector
from src.perception.basket_detector import BasketDetector

# Load config
config_path = os.path.join(project_root, 'config.yaml')
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

print("✓ Modules imported")

## 2. Understanding HSV Color Space

**Why HSV instead of BGR?**
- **H**ue: The actual color (0-180 in OpenCV)
- **S**aturation: Color intensity (0-255)
- **V**alue: Brightness (0-255)

HSV is better for color detection because:
- Hue separates color from brightness
- Works better under varying lighting
- Easier to define color ranges

In [ ]:
# Initialize camera
camera = CameraController(config)
camera.initialize()
camera.center()
camera.look_down()

# Capture frame
frame = camera.read()

if frame is not None:
    # Convert to HSV
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    
    # Split channels
    h, s, v = cv2.split(hsv)
    
    print("Original BGR image:")
    _, buffer = cv2.imencode('.jpg', frame)
    display(Image(data=buffer.tobytes()))
    
    print("\nHue channel (color):")
    _, buffer = cv2.imencode('.jpg', h)
    display(Image(data=buffer.tobytes()))
    
    print("Saturation channel (intensity):")
    _, buffer = cv2.imencode('.jpg', s)
    display(Image(data=buffer.tobytes()))
    
    print("Value channel (brightness):")
    _, buffer = cv2.imencode('.jpg', v)
    display(Image(data=buffer.tobytes()))

## 3. Ball Detection Explained

### How it works:
1. Convert frame to HSV
2. Create mask for each color range
3. Find contours in mask
4. Filter by circularity and size
5. Calculate centroid and distance

In [ ]:
# Initialize ball detector
ball_detector = BallDetector(config)

# Show color ranges
print("Ball Color Ranges (HSV):")
print("=" * 50)
for color_name, color_range in ball_detector.color_ranges.items():
    lower = color_range['hsv_lower']
    upper = color_range['hsv_upper']
    print(f"\n{color_name.upper()}:")
    print(f"  Lower: H={lower[0]:3d}, S={lower[1]:3d}, V={lower[2]:3d}")
    print(f"  Upper: H={upper[0]:3d}, S={upper[1]:3d}, V={upper[2]:3d}")

### Step-by-step Ball Detection

In [ ]:
frame = camera.read()

if frame is not None:
    # Step 1: Convert to HSV
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    
    # Step 2: Create blue ball mask
    blue_range = ball_detector.color_ranges['blue']
    blue_mask = cv2.inRange(hsv, blue_range['hsv_lower'], blue_range['hsv_upper'])
    
    print("Step 1 - Original frame:")
    _, buffer = cv2.imencode('.jpg', frame)
    display(Image(data=buffer.tobytes()))
    
    print("\nStep 2 - Blue color mask (white = detected):")
    _, buffer = cv2.imencode('.jpg', blue_mask)
    display(Image(data=buffer.tobytes()))
    
    # Step 3: Morphological operations (reduce noise)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    blue_mask_clean = cv2.erode(blue_mask, kernel, iterations=1)
    blue_mask_clean = cv2.dilate(blue_mask_clean, kernel, iterations=2)
    
    print("\nStep 3 - Cleaned mask (noise removed):")
    _, buffer = cv2.imencode('.jpg', blue_mask_clean)
    display(Image(data=buffer.tobytes()))
    
    # Step 4: Find contours
    contours, _ = cv2.findContours(blue_mask_clean, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    # Draw contours
    contour_frame = frame.copy()
    cv2.drawContours(contour_frame, contours, -1, (0, 255, 0), 2)
    
    print(f"\nStep 4 - Found {len(contours)} contours:")
    _, buffer = cv2.imencode('.jpg', contour_frame)
    display(Image(data=buffer.tobytes()))
    
    # Step 5: Full detection with filtering
    balls = ball_detector.detect(frame)
    result_frame = ball_detector.draw_detections(frame, balls)
    
    print(f"\nStep 5 - Final detection ({len(balls)} balls after filtering):")
    for ball in balls:
        color, centroid, distance, area = ball
        print(f"  {color}: {distance:.0f}cm at {centroid}, area={area:.0f}px")
    
    _, buffer = cv2.imencode('.jpg', result_frame)
    display(Image(data=buffer.tobytes()))

### Distance Estimation

**Formula:** `distance = (known_diameter × focal_length) / pixel_diameter`

- Known diameter: 3.5 cm (bottle cap size)
- Focal length: Calibrated value (~300 pixels)
- Pixel diameter: Calculated from contour area

In [ ]:
print("Distance Estimation Parameters:")
print(f"  Known ball diameter: {ball_detector.known_ball_diameter_cm} cm")
print(f"  Focal length: {ball_detector.focal_length_px} pixels")
print(f"  Min area threshold: {ball_detector.min_area} pixels²")

# Example calculation
example_area = 500  # pixels²
pixel_diameter = 2 * np.sqrt(example_area / np.pi)
distance = (ball_detector.known_ball_diameter_cm * ball_detector.focal_length_px) / pixel_diameter

print(f"\nExample:")
print(f"  If ball area = {example_area} px²")
print(f"  Then pixel diameter = {pixel_diameter:.1f} px")
print(f"  Estimated distance = {distance:.1f} cm")

## 4. Interactive HSV Tuning Tool

Use this to find optimal HSV ranges for different lighting conditions.

In [ ]:
# Create sliders
h_min = widgets.IntSlider(value=100, min=0, max=180, description='H min:')
h_max = widgets.IntSlider(value=130, min=0, max=180, description='H max:')
s_min = widgets.IntSlider(value=150, min=0, max=255, description='S min:')
s_max = widgets.IntSlider(value=255, min=0, max=255, description='S max:')
v_min = widgets.IntSlider(value=50, min=0, max=255, description='V min:')
v_max = widgets.IntSlider(value=255, min=0, max=255, description='V max:')

output_widget = widgets.Output()

def update_mask(change):
    with output_widget:
        output_widget.clear_output(wait=True)
        
        frame = camera.read()
        if frame is not None:
            hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
            
            lower = np.array([h_min.value, s_min.value, v_min.value])
            upper = np.array([h_max.value, s_max.value, v_max.value])
            
            mask = cv2.inRange(hsv, lower, upper)
            result = cv2.bitwise_and(frame, frame, mask=mask)
            
            # Show side by side
            combined = np.hstack([frame, result])
            _, buffer = cv2.imencode('.jpg', combined)
            display(Image(data=buffer.tobytes()))
            
            print(f"HSV Range: [{h_min.value}, {s_min.value}, {v_min.value}] to [{h_max.value}, {s_max.value}, {v_max.value}]")

# Attach observers
for slider in [h_min, h_max, s_min, s_max, v_min, v_max]:
    slider.observe(update_mask, names='value')

display(widgets.VBox([
    widgets.HTML("<h3>Adjust sliders to tune color detection:</h3>"),
    h_min, h_max, s_min, s_max, v_min, v_max,
    output_widget
]))

# Initial update
update_mask(None)

## 5. Obstacle Detection Explained

### Two types of obstacles:
1. **Yellow boundary tape** - Arena edges
2. **Physical obstacles** - Detected via edge detection

### Region of Interest (ROI):
- **Boundary ROI**: Bottom 30% of frame
- **Obstacle ROI**: Middle section (15% to 65%)

In [ ]:
# Initialize obstacle detector
obstacle_detector = ObstacleDetector(config)

frame = camera.read()

if frame is not None:
    h, w = frame.shape[:2]
    
    # Draw ROIs
    roi_frame = frame.copy()
    
    # Boundary ROI (bottom)
    boundary_top = int(h * (1 - obstacle_detector.bottom_roi_frac))
    cv2.rectangle(roi_frame, (0, boundary_top), (w, h), (0, 255, 255), 2)
    cv2.putText(roi_frame, "Boundary ROI", (10, boundary_top - 5),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)
    
    # Obstacle ROI (middle)
    obs_top = int(h * obstacle_detector.front_roi_top_frac)
    obs_bottom = int(h * obstacle_detector.front_roi_bottom_frac)
    cv2.rectangle(roi_frame, (0, obs_top), (w, obs_bottom), (255, 0, 255), 2)
    cv2.putText(roi_frame, "Obstacle ROI", (10, obs_top - 5),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 255), 2)
    
    print("Regions of Interest:")
    _, buffer = cv2.imencode('.jpg', roi_frame)
    display(Image(data=buffer.tobytes()))
    
    print(f"\nBoundary ROI: Bottom {obstacle_detector.bottom_roi_frac*100:.0f}% of frame")
    print(f"Obstacle ROI: {obstacle_detector.front_roi_top_frac*100:.0f}% to {obstacle_detector.front_roi_bottom_frac*100:.0f}% of frame")

### Yellow Boundary Detection Process

In [ ]:
frame = camera.read()

if frame is not None:
    h, w = frame.shape[:2]
    
    # Extract boundary ROI
    roi_top = int(h * (1 - obstacle_detector.bottom_roi_frac))
    roi = frame[roi_top:h, 0:w]
    
    print("Step 1 - Boundary ROI:")
    _, buffer = cv2.imencode('.jpg', roi)
    display(Image(data=buffer.tobytes()))
    
    # Convert to HSV and detect yellow
    hsv = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)
    yellow_mask = cv2.inRange(hsv, obstacle_detector.yellow_lower, obstacle_detector.yellow_upper)
    
    print("\nStep 2 - Yellow mask:")
    _, buffer = cv2.imencode('.jpg', yellow_mask)
    display(Image(data=buffer.tobytes()))
    
    # Count pixels
    yellow_pixels = cv2.countNonZero(yellow_mask)
    threshold = obstacle_detector.threshold_px
    
    print(f"\nYellow pixels: {yellow_pixels}")
    print(f"Threshold: {threshold}")
    print(f"Boundary detected: {yellow_pixels > threshold}")

### Edge Detection for Obstacles

In [ ]:
frame = camera.read()

if frame is not None:
    h, w = frame.shape[:2]
    
    # Extract obstacle ROI
    roi_top = int(h * obstacle_detector.front_roi_top_frac)
    roi_bottom = int(h * obstacle_detector.front_roi_bottom_frac)
    roi = frame[roi_top:roi_bottom, 0:w]
    
    print("Step 1 - Obstacle ROI:")
    _, buffer = cv2.imencode('.jpg', roi)
    display(Image(data=buffer.tobytes()))
    
    # Convert to grayscale
    gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
    
    print("\nStep 2 - Grayscale:")
    _, buffer = cv2.imencode('.jpg', gray)
    display(Image(data=buffer.tobytes()))
    
    # Edge detection
    edges = cv2.Canny(gray, 50, 150)
    
    print("\nStep 3 - Edges (Canny):")
    _, buffer = cv2.imencode('.jpg', edges)
    display(Image(data=buffer.tobytes()))
    
    # Count edge pixels
    edge_pixels = cv2.countNonZero(edges)
    threshold = obstacle_detector.edge_threshold
    
    print(f"\nEdge pixels: {edge_pixels}")
    print(f"Threshold: {threshold}")
    print(f"Obstacle detected: {edge_pixels > threshold}")

## 6. Complete Detection Demo

In [ ]:
# Initialize all detectors
ball_detector = BallDetector(config)
obstacle_detector = ObstacleDetector(config)
basket_detector = BasketDetector(config)

frame = camera.read()

if frame is not None:
    # Run all detections
    balls = ball_detector.detect(frame)
    obstacle = obstacle_detector.detect_combined(frame)
    basket = basket_detector.detect(frame)
    
    # Draw all overlays
    result = ball_detector.draw_detections(frame, balls)
    result = obstacle_detector.draw_detections(result, obstacle)
    result = basket_detector.draw_detection(result, basket)
    
    print("Complete Detection Results:")
    print("=" * 50)
    print(f"\nBalls: {len(balls)}")
    for ball in balls:
        color, centroid, distance, area = ball
        print(f"  - {color}: {distance:.0f}cm at {centroid}")
    
    print(f"\nBoundary: {'YES' if obstacle['boundary_detected'] else 'NO'}")
    if obstacle['boundary_detected']:
        print(f"  Turn: {obstacle['turn_direction']}")
    
    print(f"\nObstacle: {'YES' if obstacle['obstacle_detected'] else 'NO'}")
    if obstacle['obstacle_detected']:
        print(f"  Turn: {obstacle['turn_direction']}")
    
    print(f"\nBasket: {'FOUND' if basket['basket_found'] else 'NOT FOUND'}")
    if basket['basket_found']:
        print(f"  Distance: {basket['distance']:.0f}cm")
        print(f"  Bearing: {basket['bearing']:.0f}°")
    
    print("\n")
    _, buffer = cv2.imencode('.jpg', result)
    display(Image(data=buffer.tobytes()))

## 7. Cleanup

In [ ]:
camera.release()
print("✓ Cleanup complete")

## Summary

You've learned:
- ✓ HSV color space and why it's better for detection
- ✓ Ball detection: color segmentation, contours, circularity filtering
- ✓ Distance estimation from pixel area
- ✓ Obstacle detection: ROIs, yellow tape, edge detection
- ✓ How to tune HSV ranges for different lighting

**Next:** Move to `03_live_detection_viewer.ipynb` to see real-time detection in action!